In [3]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.ops as ops
from torch.utils.data import DataLoader
from torchsummary import summary

from data_preprocessing import FlagOnGroundDataset

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [4]:
collab = False
if collab:
    from google.colab import drive

    drive.mount('/content/drive')
    folder_path = 'drive/MyDrive/ai_data/data'
else:
    folder_path = '../data'

In [5]:
dataset = FlagOnGroundDataset(os.path.join(folder_path, 'flags/'), os.path.join(folder_path, 'desert/'), False)
img, rect = dataset[0]

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [7]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.double_conv(x)
        return x

class SmallDetectorNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layer = nn.Sequential(
            DoubleConv(3, 64),
            nn.MaxPool2d(2),

            DoubleConv(64, 128),
            nn.MaxPool2d(2),

            DoubleConv(128, 64),
            nn.MaxPool2d(2),

            DoubleConv(64, 32),
            nn.MaxPool2d(2),

            DoubleConv(32, 16),
        )

        self.head = nn.Sequential(
            nn.Conv2d(16, 1, kernel_size=3, stride=1, padding=1),
            # nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.head(x)

        return x


def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


model = SmallDetectorNetwork()
model.apply(init_weights)

# model.load_state_dict(torch.load(os.path.join(folder_path, 'checkpoints/custom_model.pt'), weights_only=True))

SmallDetectorNetwork(
  (conv_layer): Sequential(
    (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): Conv2d(8, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU()
  )
  (head): Sequential(
    (0): Conv2d(4, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)

In [10]:
opt = optim.Adam(model.parameters(), lr=1e-2)
epochs = 9999

for epoch in range(epochs):
    model = model.to(device)
    for img, target_rect in dataloader:
        x1 = target_rect[:, 0]
        y1 = target_rect[:, 1]
        x2 = target_rect[:, 2]
        y2 = target_rect[:, 3]

        mask = torch.zeros((img.shape[0], 1, img.shape[2], img.shape[3]), device=device)

        for i in range(img.shape[0]):
            mask[i, :, y1[i]:y2[i], x1[i]:x2[i]] = 1

        prediction = model(img)
        loss = ops.focal_loss.sigmoid_focal_loss(prediction, mask, alpha=0.25, gamma=3.0).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()
        
    print(f"Epoch: {epoch}, loss: {loss.item()}")
    model.to(torch.device('cpu'))
    torch.save(model.state_dict(), os.path.join(folder_path, 'checkpoints/custom_model.pt'))
    

KeyboardInterrupt: 